In [6]:
# [셀 1] 설치 — transformers가 이미 충분히 새 버전이면 pip이 건너뜀 (불필요한 재설치 없음)
# SigLIP2는 transformers>=4.49 부터 내장으로 알고 있음 (정확한 최소 버전은 확신 없음)
!pip install -q "transformers>=4.49"
# torch / numpy / pandas / pillow 는 Kaggle 기본 이미지에 있으므로 설치하지 않음

In [7]:
# [셀 2] 설정 / import
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # T4 x2 중 GPU 1장만 사용 (반드시 torch import 전에, 커널 재시작 직후 실행)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys, glob, json, math, time, gc, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# ===== 여기만 바꾸면 됨 =====
MODEL_NAME = "google/siglip2-so400m-patch14-384"   # 대회에서 사용할 모델
DEVICE = "cuda:0"
WORK = "/kaggle/working"

TEXT_BS = 128          # 텍스트 배치 (max_len 64라 OOM 위험 거의 없음)
IMG_BS = 32            # 이미지 배치 (so400m-384 fp16 기준 T4에서 여유 있음 (T4 16GB; 실측 전이라 확신 없음), OOM 나면 자동으로 반씩 쪼갬)
CHUNK_IMG = 250        # 이 개수마다 중간 임베딩을 디스크에 저장 (세션 끊김 복구용)
CHUNK_TXT = 500
NUM_WORKERS = 2        # DataLoader 워커 수 (Kaggle CPU 4코어 기준)
FORCE_RECOMPUTE = False   # True면 캐시를 무시하고 다시 계산 (속도 측정을 다시 하고 싶을 때)

IMG_COL = None         # None이면 자동 탐지, 자동 탐지가 틀리면 컬럼명을 직접 지정
CAP_COL = None
# ===========================

assert torch.cuda.is_available(), "GPU가 꺼져 있음: Settings > Accelerator 확인"
print("torch", torch.__version__, "| gpu:", torch.cuda.get_device_name(0), "| visible gpus:", torch.cuda.device_count())
import transformers
print("transformers", transformers.__version__)

torch 2.11.0+cu130 | gpu: Tesla T4 | visible gpus: 1
transformers 5.20.2


In [8]:
# [셀 3] 입력 경로 확인
!ls -R /kaggle/input | head -30

/kaggle/input:
datasets

/kaggle/input/datasets:
dahyvvn

/kaggle/input/datasets/dahyvvn:
haikuthon

/kaggle/input/datasets/dahyvvn/haikuthon:
images
prompts.csv

/kaggle/input/datasets/dahyvvn/haikuthon/images:
0000.png
0001.png
0002.png
0003.png
0004.png
0005.png
0006.png
0007.png
0008.png
0009.png
0010.png
0011.png
0012.png
0013.png
0014.png
0015.png
ls: write error: Broken pipe


In [9]:
# [셀 4] prompts.csv 로드 + 이미지/캡션 컬럼 연결 확인
CSV_PATHS = sorted(glob.glob("/kaggle/input/**/prompts.csv", recursive=True))
assert CSV_PATHS, "prompts.csv를 /kaggle/input 아래에서 못 찾음 -> 위 ls 결과의 실제 경로 확인"
CSV_PATH = CSV_PATHS[0]
if len(CSV_PATHS) > 1:
    print("WARNING: prompts.csv가 여러 개 있음:", CSV_PATHS)
ROOT = os.path.dirname(CSV_PATH)
IMG_DIR = os.path.join(ROOT, "images")
if not os.path.isdir(IMG_DIR):
    cands = [p for p in glob.glob("/kaggle/input/**/images", recursive=True) if os.path.isdir(p)]
    assert cands, "images/ 폴더를 못 찾음"
    IMG_DIR = cands[0]
print("CSV_PATH:", CSV_PATH)
print("IMG_DIR :", IMG_DIR)

df = pd.read_csv(CSV_PATH)          # 주의: 이후 df의 행 순서/행 수를 바꾸지 말 것 (split 재현성이 여기에 달려 있음)
print("columns:", list(df.columns))
print("rows   :", len(df))
print(df.dtypes.to_string())

# 이미지 파일 인덱스 (파일명 / 확장자 뺀 이름 둘 다로 찾을 수 있게)
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
by_name, by_stem = {}, {}
for dp, _, fns in os.walk(IMG_DIR):
    for fn in fns:
        if os.path.splitext(fn)[1].lower() in IMG_EXTS:
            p = os.path.join(dp, fn)
            by_name[fn] = p
            by_stem[os.path.splitext(fn)[0]] = p
print("image files found:", len(by_name))

from collections import Counter
STEM_W = Counter(len(k) for k in by_stem).most_common(1)[0][0]   # 파일명(확장자 제외) 자릿수, 예: 0000.png -> 4

def resolve(v):
    b = os.path.basename(str(v).strip().replace("\\", "/"))
    stem = os.path.splitext(b)[0]
    # id가 정수(7)인데 파일명이 0패딩(0007.png)인 경우까지 처리
    return by_name.get(b) or by_stem.get(stem) or by_stem.get(stem.zfill(STEM_W))

# 이미지 컬럼 = 값이 실제 이미지 파일로 가장 많이 연결되는 컬럼
if IMG_COL is None:
    scores = {c: float(df[c].map(lambda v: resolve(v) is not None).mean()) for c in df.columns}
    IMG_COL = max(scores, key=scores.get)
    print("image-col match ratio:", {k: round(v, 3) for k, v in scores.items()})
    assert scores[IMG_COL] > 0.95, "이미지 컬럼 자동 탐지 실패 -> IMG_COL을 직접 지정"
# 캡션 컬럼 = (이미지 컬럼 제외) 문자열 컬럼 중 평균 길이가 가장 긴 컬럼
if CAP_COL is None:
    str_cols = [c for c in df.columns if c != IMG_COL and (df[c].dtype == object or pd.api.types.is_string_dtype(df[c]))]   # pandas 2.x(object)/3.x(str) 모두 대응
    assert str_cols, "캡션 후보 컬럼 없음 -> CAP_COL을 직접 지정"
    CAP_COL = max(str_cols, key=lambda c: df[c].astype(str).str.len().mean())
print(f"==> IMG_COL = {IMG_COL!r}, CAP_COL = {CAP_COL!r}")

df["_img_path"] = df[IMG_COL].map(resolve)      # 컬럼 추가만 함 (행 순서 불변)
assert df["_img_path"].notna().all(), "경로를 못 찾은 이미지가 있음"
if df[CAP_COL].isna().any():
    print("WARNING: 캡션 NaN", int(df[CAP_COL].isna().sum()), "개 (빈 문자열로 처리)")
print("duplicated captions:", int(df[CAP_COL].duplicated().sum()), "| duplicated image files:", int(df["_img_path"].duplicated().sum()))

print("\n--- 샘플 3개 ---")
for i in range(3):
    p = df.loc[i, "_img_path"]
    w, h = Image.open(p).size
    print(f"[{i}] file={os.path.basename(p)} ({IMG_COL}={df.loc[i, IMG_COL]!r}) | size={w}x{h} | {os.path.getsize(p)/1024:.0f} KB")
    print("    caption:", str(df.loc[i, CAP_COL])[:600])

CSV_PATH: /kaggle/input/datasets/dahyvvn/haikuthon/prompts.csv
IMG_DIR : /kaggle/input/datasets/dahyvvn/haikuthon/images
columns: ['id', 'prompt']
rows   : 4000
id         int64
prompt    object
image files found: 4000
image-col match ratio: {'id': 1.0, 'prompt': 0.0}
==> IMG_COL = 'id', CAP_COL = 'prompt'
duplicated captions: 0 | duplicated image files: 0

--- 샘플 3개 ---
[0] file=0000.png (id=np.int64(0)) | size=768x768 | 954 KB
    caption: One person is lying on an airport apron. One scarf is solid-colored and gray. The person is wearing the gray scarf. One scarf is polka-dotted and orange. The person is above the orange scarf. The scene is a photograph from a front view with side lighting.
[1] file=0001.png (id=np.int64(1)) | size=768x768 | 1175 KB
    caption: One square yellow drinking glass is above one white ear of corn and above one orange pumpkin in a kitchen. The scene is a colored-pencil drawing in side view with warm indoor lighting.
[2] file=0002.png (id=np.int64(2)) | siz

In [10]:
# [셀 5] validation 분리 (팀원 모두 동일 결과가 나와야 하는 고정 방식)
idx = np.random.RandomState(42).permutation(len(df))
val_idx = sorted(idx[:500].tolist())
train_idx = sorted(idx[500:].tolist())

with open(f"{WORK}/val_idx.json", "w") as f:
    json.dump(val_idx, f)

val_md5 = hashlib.md5(json.dumps(val_idx).encode()).hexdigest()
print(f"len(val)={len(val_idx)}, len(train)={len(train_idx)}, val_idx md5={val_md5}   <- 팀원끼리 이 값이 같은지 확인")
print("val_idx[:10]:", val_idx[:10])
for i in val_idx[:10]:
    print(f"  {i:5d} | {str(df.loc[i, CAP_COL])[:50]}")

def prep_text(s):
    # SigLIP2는 소문자 텍스트로 학습됨 (공식 문서) -> 소문자로 통일
    return "" if pd.isna(s) else str(s).strip().lower()

val_texts = [prep_text(df.loc[i, CAP_COL]) for i in val_idx]
val_paths = [df.loc[i, "_img_path"] for i in val_idx]

# 임베딩 캐시 폴더: 모델 + val 분할이 바뀌면 자동으로 다른 폴더를 씀
CACHE = f"{WORK}/emb_cache/{MODEL_NAME.replace('/', '__')}/val_{val_md5[:8]}"
os.makedirs(CACHE, exist_ok=True)
print("cache dir:", CACHE)

len(val)=500, len(train)=3500, val_idx md5=fd8a1a387fbe6bbb3bf9d0f0d6b773ad   <- 팀원끼리 이 값이 같은지 확인
val_idx[:10]: [8, 12, 14, 17, 23, 26, 32, 33, 44, 45]
      8 | One picture frame, with one yellow slice of pizza 
     12 | One owl, black and standing, appears with one blac
     14 | On a farm, there are one watering can, two cars th
     17 | At a farm, one person riding one horse is sitting,
     23 | Two beige goats to the right of one standing zebra
     26 | To the left of one baseball bat, yellow, is one ca
     32 | One person is standing and wearing one scarf, whic
     33 | In a park, one person, kneeling and wearing one he
     44 | One gardening trowel, white, is to the left of one
     45 | A studio with one plain backdrop is shown in a 3D 
cache dir: /kaggle/working/emb_cache/google__siglip2-so400m-patch14-384/val_fd8a1a38


In [11]:
# [셀 6] 모델 로드 (fp16) + 가중치 용량
from transformers import AutoModel, AutoProcessor

# SigLIP2는 transformers 내장 클래스라 trust_remote_code 불필요 (다른 모델이면 필요할 수 있음 -> 그때만 True)
processor = AutoProcessor.from_pretrained(MODEL_NAME)
# 주의: 구버전/신버전 모두에서 fp16으로 확실히 올라가도록 torch_dtype 사용 (신버전에서는 deprecation 경고만 뜰 수 있음)
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE).eval()
assert next(model.parameters()).dtype == torch.float16, "fp16 로드 실패"

tokenizer = processor.tokenizer
image_processor = processor.image_processor

n_params = sum(p.numel() for p in model.parameters())
fp16_bytes = n_params * 2
print(f"params: {n_params/1e6:.1f} M")
print(f"fp16 weights: {fp16_bytes/1e9:.3f} GB ({fp16_bytes/2**30:.3f} GiB)   | 제출 제한 5GB 대비 {fp16_bytes/5e9*100:.0f}%")
for name in ("text_model", "vision_model"):
    m = getattr(model, name, None)
    if m is not None:
        print(f"  - {name}: {sum(p.numel() for p in m.parameters())*2/1e9:.3f} GB (fp16)")

# 텍스트 최대 길이: SigLIP2는 max_length=64, padding="max_length"로 학습됨 (공식 문서)
MAX_LEN = int(getattr(model.config.text_config, "max_position_embeddings", 64))
print("text MAX_LEN:", MAX_LEN)
print("image processor:", type(image_processor).__name__)

'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/google/siglip2-so400m-patch14-384/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/google/siglip2-so400m-patch14-384/resolve/main/processor_config.json
Retrying in 2s [Retry 2/5].


preprocessor_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/559 [00:00<?, ?B/s]

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 34.4MB            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.54GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/888 [00:00<?, ?it/s]

params: 1136.0 M
fp16 weights: 2.272 GB (2.116 GiB)   | 제출 제한 5GB 대비 45%
  - text_model: 1.416 GB (fp16)
  - vision_model: 0.856 GB (fp16)
text MAX_LEN: 64
image processor: SiglipImageProcessor


In [12]:
# [셀 7] 텍스트 토큰 길이 분포 (truncation 없이 원래 길이를 측정)
def len_stats(name, texts):
    ids = tokenizer(texts, truncation=False, padding=False)["input_ids"]   # 참고: EOS/BOS를 붙이는지는 토크나이저 버전에 따라 다름 (확신 없음)
    lens = np.array([len(x) for x in ids])
    over = lens > MAX_LEN
    lost = np.clip(lens - MAX_LEN, 0, None).sum() / lens.sum()
    print(f"[{name}] n={len(lens)} | max={lens.max()} | mean={lens.mean():.1f} | median={np.median(lens):.0f} | "
          f"p90={np.percentile(lens, 90):.0f} | p99={np.percentile(lens, 99):.0f}")
    print(f"        MAX_LEN({MAX_LEN})보다 긴 캡션 비율 = {over.mean()*100:.1f}% | 잘려나가는 토큰 비율(전체 토큰 대비) = {lost*100:.1f}%")
    return lens

all_texts = [prep_text(s) for s in df[CAP_COL]]
_ = len_stats("val 500", val_texts)
_ = len_stats("전체 4000", all_texts)

[val 500] n=500 | max=81 | mean=44.1 | median=43 | p90=55 | p99=68
        MAX_LEN(64)보다 긴 캡션 비율 = 2.0% | 잘려나가는 토큰 비율(전체 토큰 대비) = 0.2%
[전체 4000] n=4000 | max=81 | mean=44.1 | median=43 | p90=55 | p99=68
        MAX_LEN(64)보다 긴 캡션 비율 = 2.1% | 잘려나가는 토큰 비율(전체 토큰 대비) = 0.2%


In [13]:
# [셀 8] 인코딩 함수 정의 (배치 처리 + no_grad + OOM 자동 분할 + 청크 단위 디스크 저장)
def to_tensor(out):
    # 버전에 따라 get_*_features가 Tensor를 주기도 하고 BaseModelOutputWithPooling을 주기도 함
    if torch.is_tensor(out):
        return out
    for k in ("pooler_output", "image_embeds", "text_embeds"):
        v = getattr(out, k, None)
        if v is not None:
            return v
    raise TypeError(f"features 출력 형식을 모름: {type(out)}")   # 확신 없음: 새 버전에서 pooler_output이 투영된 임베딩인지 직접 확인 권장

# ---------- 텍스트 ----------
def tokenize(texts):
    try:   # 공식 사용법: processor(text=..., padding="max_length", max_length=64, truncation=True)
        return processor(text=texts, padding="max_length", max_length=MAX_LEN, truncation=True, return_tensors="pt")
    except TypeError:
        return tokenizer(texts, padding="max_length", max_length=MAX_LEN, truncation=True, return_tensors="pt")

TXT_STATS = {"n": 0, "sec": 0.0}

@torch.no_grad()
def txt_chunk(texts):
    feats = []
    for bi, i in enumerate(range(0, len(texts), TEXT_BS)):
        inputs = {k: v.to(DEVICE) for k, v in tokenize(texts[i:i + TEXT_BS]).items()}
        # 어떤 키(input_ids / attention_mask)가 모델에 들어가는지는 processor 출력 그대로 따름 (확신 없음)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        out = model.get_text_features(**inputs)
        f = F.normalize(to_tensor(out).float(), dim=-1)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        if bi > 0:                                   # 첫 배치(워밍업) 제외
            TXT_STATS["n"] += len(inputs["input_ids"]); TXT_STATS["sec"] += dt
        feats.append(f.cpu())
    return torch.cat(feats)

# ---------- 이미지 ----------
class ImgDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        with Image.open(self.paths[i]) as im:
            return im.convert("RGB")

def collate_imgs(imgs):
    return dict(image_processor(images=imgs, return_tensors="pt"))   # NaFlex면 pixel_attention_mask, spatial_shapes도 같이 나옴

@torch.no_grad()
def forward_img(batch):
    oom = False
    try:
        inp = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
        inp["pixel_values"] = inp["pixel_values"].half()
        out = model.get_image_features(**inp)
        return F.normalize(to_tensor(out).float(), dim=-1).cpu()
    except torch.cuda.OutOfMemoryError:
        oom = True
    # OOM이면 배치를 반으로 쪼개서 재시도 (except 블록 밖에서 해야 메모리가 풀림)
    inp = out = None
    torch.cuda.empty_cache()
    bs = batch["pixel_values"].shape[0]
    if bs == 1:
        raise RuntimeError("배치 1장도 OOM")
    h = bs // 2
    return torch.cat([forward_img({k: v[:h] for k, v in batch.items()}),
                      forward_img({k: v[h:] for k, v in batch.items()})])

IMG_STATS = {"n": 0, "wall": 0.0, "gpu": 0.0}

def img_chunk(paths):
    loader = DataLoader(ImgDS(paths), batch_size=IMG_BS, shuffle=False, num_workers=NUM_WORKERS,
                        collate_fn=collate_imgs, pin_memory=True)
    feats = []
    torch.cuda.synchronize(); t_prev = time.perf_counter()
    for bi, batch in enumerate(loader):
        bs = batch["pixel_values"].shape[0]
        torch.cuda.synchronize(); t_g = time.perf_counter()
        feats.append(forward_img(batch))
        torch.cuda.synchronize(); t_now = time.perf_counter()
        if bi > 0:      # 청크마다 첫 배치(워커 기동 + cudnn 워밍업) 제외
            IMG_STATS["n"] += bs
            IMG_STATS["wall"] += t_now - t_prev      # 데이터 로딩 + 전처리 + GPU 포함 (end-to-end)
            IMG_STATS["gpu"] += t_now - t_g          # GPU 순수 시간(H2D 복사 포함)
        t_prev = t_now
    return torch.cat(feats)

# ---------- 청크 캐시 ----------
def run_chunked(n, chunk, tag, fn, force=False):
    outs, computed = [], 0
    for ci, s in enumerate(range(0, n, chunk)):
        e = min(s + chunk, n)
        fpath = f"{CACHE}/{tag}_{ci:04d}.pt"
        if os.path.exists(fpath) and not force:
            t = torch.load(fpath, map_location="cpu")
            if t.shape[0] == e - s:
                outs.append(t)
                print(f"  [{tag}] chunk {ci} 캐시 사용 ({s}:{e})")
                continue
        t = fn(s, e)
        torch.save(t, fpath + ".tmp"); os.replace(fpath + ".tmp", fpath)   # 저장 도중 끊겨도 깨진 파일이 남지 않게
        outs.append(t); computed += e - s
        print(f"  [{tag}] chunk {ci} 계산·저장 완료 ({s}:{e})")
    return torch.cat(outs), computed

In [14]:
# [셀 9] 인코딩 실행 (세션이 끊기면 셀 2~6을 다시 실행한 뒤 이 셀부터 다시 돌리면 저장된 청크는 건너뜀)
t0 = time.time()
txt_emb, n_t = run_chunked(len(val_texts), CHUNK_TXT, f"txt_L{MAX_LEN}",
                           lambda s, e: txt_chunk(val_texts[s:e]), FORCE_RECOMPUTE)
print(f"text emb: {tuple(txt_emb.shape)} (새로 계산 {n_t}개), {time.time()-t0:.1f}s")

t0 = time.time()
img_emb, n_i = run_chunked(len(val_paths), CHUNK_IMG, "img",
                           lambda s, e: img_chunk(val_paths[s:e]), FORCE_RECOMPUTE)
print(f"image emb: {tuple(img_emb.shape)} (새로 계산 {n_i}개), {time.time()-t0:.1f}s")

assert txt_emb.shape[0] == img_emb.shape[0] == 500
print("norm check (should be ~1):", float(txt_emb.norm(dim=-1).mean()), float(img_emb.norm(dim=-1).mean()))

  [txt_L64] chunk 0 계산·저장 완료 (0:500)
text emb: (500, 1152) (새로 계산 500개), 2.5s
  [img] chunk 0 계산·저장 완료 (0:250)
  [img] chunk 1 계산·저장 완료 (250:500)
image emb: (500, 1152) (새로 계산 500개), 39.5s
norm check (should be ~1): 1.0 1.0


In [15]:
# [셀 10] zero-shot 평가: 텍스트 500개 -> 이미지 500장 gallery (정답 = 같은 인덱스)
def retrieval_metrics(sims):
    # sims[i, j] = 텍스트 i vs 이미지 j 유사도, 정답은 대각선. 순위 r = 1 + (정답보다 점수가 '엄격히' 높은 이미지 수)
    sims = np.asarray(sims, dtype=np.float32)
    pos = np.diag(sims)[:, None]
    ranks = 1 + (sims > pos).sum(axis=1)
    ties = (sims == pos).sum(axis=1) - 1           # 정답과 점수가 정확히 같은 오답 수 (있으면 낙관적 순위가 됨)
    recall1 = (ranks == 1).astype(np.float64)
    ndcg5 = np.where(ranks <= 5, 1.0 / np.log2(ranks + 1.0), 0.0)
    return dict(recall1=recall1.mean(), ndcg5=ndcg5.mean(), score=(0.5 * (recall1 + ndcg5)).mean(),
                recall5=(ranks <= 5).mean(), recall10=(ranks <= 10).mean(), mrr=(1.0 / ranks).mean(),
                median_rank=float(np.median(ranks)), n_ties=int((ties > 0).sum())), ranks

sims = (txt_emb @ img_emb.T).numpy()
m, ranks = retrieval_metrics(sims)

print("=" * 60)
print(f"model: {MODEL_NAME}  | val queries: {len(ranks)}, gallery: {sims.shape[1]}")
print(f"Recall@1     : {m['recall1']:.4f}")
print(f"NDCG@5       : {m['ndcg5']:.4f}")
print(f"FINAL SCORE  : {m['score']:.4f}   (= mean of 0.5*(Recall@1 + NDCG@5))")
print(f"(참고) Recall@5 {m['recall5']:.4f} | Recall@10 {m['recall10']:.4f} | MRR {m['mrr']:.4f} | median rank {m['median_rank']:.0f} | 동점 쿼리 {m['n_ties']}")
print("=" * 60)

# ---- 임베딩 저장 (L2 정규화된 float32; 이후 재사용) ----
tag = MODEL_NAME.replace("/", "__")
torch.save({"emb": img_emb, "idx": val_idx}, f"{WORK}/val_img_emb_{tag}.pt")
torch.save({"emb": txt_emb, "idx": val_idx}, f"{WORK}/val_txt_emb_{tag}.pt")
np.save(f"{WORK}/val_img_emb_{tag}.npy", img_emb.numpy())
np.save(f"{WORK}/val_txt_emb_{tag}.npy", txt_emb.numpy())
np.save(f"{WORK}/val_ranks_{tag}.npy", ranks)
with open(f"{WORK}/val_eval_{tag}.json", "w") as f:
    json.dump({"model": MODEL_NAME, "max_len": MAX_LEN, "val_md5": val_md5,
               **{k: float(v) for k, v in m.items()}}, f, indent=2)
print("saved to", WORK)

model: google/siglip2-so400m-patch14-384  | val queries: 500, gallery: 500
Recall@1     : 0.7640
NDCG@5       : 0.9065
FINAL SCORE  : 0.8353   (= mean of 0.5*(Recall@1 + NDCG@5))
(참고) Recall@5 1.0000 | Recall@10 1.0000 | MRR 0.8740 | median rank 1 | 동점 쿼리 0
saved to /kaggle/working


In [16]:
# [셀 11] 속도 / 추정 소요 시간 / 메모리
LIMIT_SEC = 15 * 60
if IMG_STATS["n"] == 0:
    print("이미지 임베딩을 전부 캐시에서 읽어서 속도를 측정하지 못함 -> FORCE_RECOMPUTE=True로 바꾸고 셀 9부터 다시 실행")
else:
    ips_e2e = IMG_STATS["n"] / IMG_STATS["wall"]
    ips_gpu = IMG_STATS["n"] / IMG_STATS["gpu"]
    est_e2e = 16000 / ips_e2e
    est_gpu = 16000 / ips_gpu
    print(f"[이미지] end-to-end(디코딩+전처리+GPU): {ips_e2e:.1f} images/sec  -> 16,000장 약 {est_e2e:.0f}s ({est_e2e/60:.1f}분)")
    print(f"[이미지] GPU 순수 추론:                 {ips_gpu:.1f} images/sec  -> 16,000장 약 {est_gpu:.0f}s ({est_gpu/60:.1f}분)")
    if ips_e2e < 0.8 * ips_gpu:
        print("   -> end-to-end가 GPU 순수보다 많이 느림: CPU 디코딩/전처리가 병목 (NUM_WORKERS 조정 / 이미지 크기 확인)")
if TXT_STATS["n"] > 0:
    tps = TXT_STATS["n"] / TXT_STATS["sec"]
    print(f"[텍스트] {tps:.0f} texts/sec -> 8,000개 약 {8000/tps:.1f}s")
if IMG_STATS["n"] > 0 and TXT_STATS["n"] > 0:
    total = est_e2e + 8000 / tps
    print(f"[합계] 이미지 16,000 + 텍스트 8,000 추정 {total:.0f}s ({total/60:.1f}분) vs 제한 {LIMIT_SEC/60:.0f}분")
    print("   * 이 수치는 Kaggle T4 실측 기준. 제출 환경(L4)은 GPU/CPU가 달라 직접 측정한 값이 아님.")
    print("   * 모델 로딩 시간, 유사도 계산/정렬, 8,000x16,000 검색 시간은 포함되지 않음 (여유를 두고 판단할 것).")
print(f"[GPU 메모리] peak allocated: {torch.cuda.max_memory_allocated()/2**30:.2f} GiB")
print(f"[가중치] fp16 {fp16_bytes/1e9:.3f} GB (5GB 제한)")

[이미지] end-to-end(디코딩+전처리+GPU): 17.5 images/sec  -> 16,000장 약 916s (15.3분)
[이미지] GPU 순수 추론:                 25.0 images/sec  -> 16,000장 약 641s (10.7분)
   -> end-to-end가 GPU 순수보다 많이 느림: CPU 디코딩/전처리가 병목 (NUM_WORKERS 조정 / 이미지 크기 확인)
[텍스트] 401 texts/sec -> 8,000개 약 19.9s
[합계] 이미지 16,000 + 텍스트 8,000 추정 935s (15.6분) vs 제한 15분
   * 이 수치는 Kaggle T4 실측 기준. 제출 환경(L4)은 GPU/CPU가 달라 직접 측정한 값이 아님.
   * 모델 로딩 시간, 유사도 계산/정렬, 8,000x16,000 검색 시간은 포함되지 않음 (여유를 두고 판단할 것).
[GPU 메모리] peak allocated: 2.75 GiB
[가중치] fp16 2.272 GB (5GB 제한)
